# 04 — Schema en etoile (Star Schema)

Ce notebook transforme les 14 tables 3NF en un schema dimensionnel (etoile) pour la BI.

**Schema :** 6 dimensions + 1 table de faits
```
         dim_date ──┐
       dim_player ──┤
         dim_team ──┤── fact_player_game
          dim_map ──┤   (grain: 1 joueur × 1 game)
          dim_car ──┤
        dim_event ──┘
```

**Pipeline ETL :**
```
01_extract              Kaggle → data/raw/*.csv
02_transform            data/raw/*.csv → data/processed/*.csv (14 tables 3NF)
03_load                 data/processed/*.csv → DuckDB (data/rl.duckdb)
04_star_schema (ici)    Tables 3NF → Star schema (6 dims + 1 fact)
```

## 1. Migrations Alembic

In [ ]:
from src.etl.load.core import apply_migrations

apply_migrations()
print('Schema a jour (14 tables 3NF + 7 tables star).')

## 2. Truncate star tables

In [ ]:
from src.etl.star.core import truncate_star_tables

truncated = truncate_star_tables()
print(f'{len(truncated)} tables star videes : {truncated}')

## 3. Peuplement des dimensions

In [ ]:
from src.etl.star.core import populate_dimensions

dim_counts = populate_dimensions()
print(f'\nTotal dimensions : {sum(dim_counts.values()):,} lignes')

## 4. Peuplement de la table de faits

In [ ]:
from src.etl.star.core import populate_fact_player_game

fact_count = populate_fact_player_game()
print(f'fact_player_game : {fact_count:,} lignes')

## 5. Verification SQL

In [ ]:
import pandas as pd
from src.database.engine import engine

# Row counts
query = """
SELECT 'dim_date' AS table_name, COUNT(*) AS rows FROM dim_date
UNION ALL SELECT 'dim_player', COUNT(*) FROM dim_player
UNION ALL SELECT 'dim_team', COUNT(*) FROM dim_team
UNION ALL SELECT 'dim_map', COUNT(*) FROM dim_map
UNION ALL SELECT 'dim_car', COUNT(*) FROM dim_car
UNION ALL SELECT 'dim_event', COUNT(*) FROM dim_event
UNION ALL SELECT 'fact_player_game', COUNT(*) FROM fact_player_game
"""

df_star = pd.read_sql(query, engine)
display(df_star)
print(f'\nTotal star schema : {df_star["rows"].sum():,} lignes')

In [ ]:
# FK integrity: 0 orphelins dans fact vs chaque dim
fk_check = """
SELECT
    'player_key orphans' AS check_name,
    COUNT(*) AS orphans
FROM fact_player_game f
LEFT JOIN dim_player dp ON f.player_key = dp.player_key
WHERE f.player_key IS NOT NULL AND dp.player_key IS NULL

UNION ALL SELECT 'team_key orphans',
    COUNT(*) FROM fact_player_game f
    LEFT JOIN dim_team dt ON f.team_key = dt.team_key
    WHERE f.team_key IS NOT NULL AND dt.team_key IS NULL

UNION ALL SELECT 'map_key orphans',
    COUNT(*) FROM fact_player_game f
    LEFT JOIN dim_map dm ON f.map_key = dm.map_key
    WHERE f.map_key IS NOT NULL AND dm.map_key IS NULL

UNION ALL SELECT 'car_key orphans',
    COUNT(*) FROM fact_player_game f
    LEFT JOIN dim_car dc ON f.car_key = dc.car_key
    WHERE f.car_key IS NOT NULL AND dc.car_key IS NULL

UNION ALL SELECT 'date_key orphans',
    COUNT(*) FROM fact_player_game f
    LEFT JOIN dim_date dd ON f.date_key = dd.date_key
    WHERE f.date_key IS NOT NULL AND dd.date_key IS NULL

UNION ALL SELECT 'event_key orphans',
    COUNT(*) FROM fact_player_game f
    LEFT JOIN dim_event de ON f.event_key = de.event_key
    WHERE f.event_key IS NOT NULL AND de.event_key IS NULL
"""

df_fk = pd.read_sql(fk_check, engine)
display(df_fk)
assert df_fk['orphans'].sum() == 0, 'FK integrity violation!'
print('FK integrity OK : 0 orphelins.')

In [ ]:
# Spot-check: comparer fact.core_goals vs stat EAV
spot_check = """
SELECT
    f.game_id, f.player_id,
    f.core_goals AS fact_goals,
    s.value AS eav_goals
FROM fact_player_game f
JOIN stat s ON s.game_id = f.game_id
    AND s.entity_id = f.player_id
    AND s.entity_type = 'player'
JOIN stat_type st ON s.type_id = st.type_id AND st.stat_name = 'core_goals'
LIMIT 10
"""

df_spot = pd.read_sql(spot_check, engine)
display(df_spot)
assert (df_spot['fact_goals'] == df_spot['eav_goals']).all(), 'Spot-check mismatch!'
print('Spot-check OK : fact.core_goals = stat EAV pour les 10 premieres lignes.')

In [ ]:
# Comparaison fact vs game_player (source)
compare = """
SELECT
    (SELECT COUNT(*) FROM game_player) AS game_player_rows,
    (SELECT COUNT(*) FROM fact_player_game) AS fact_rows
"""
df_cmp = pd.read_sql(compare, engine)
display(df_cmp)
print(f'fact_player_game ({df_cmp["fact_rows"].iloc[0]:,}) ≈ game_player ({df_cmp["game_player_rows"].iloc[0]:,})')

## 6. Exemples de requetes BI

In [ ]:
# Taux de victoire par voiture (top 10)
q_winrate_car = """
SELECT
    dc.car_name,
    COUNT(*) AS games,
    ROUND(100.0 * SUM(CASE WHEN f.winner THEN 1 ELSE 0 END) / COUNT(*), 2) AS win_rate
FROM fact_player_game f
JOIN dim_car dc ON f.car_key = dc.car_key
GROUP BY dc.car_name
HAVING COUNT(*) >= 100
ORDER BY win_rate DESC
LIMIT 10
"""

print('--- Win rate par voiture (min 100 games) ---')
display(pd.read_sql(q_winrate_car, engine))

In [ ]:
# Score moyen par event_tier
q_score_tier = """
SELECT
    de.event_tier,
    COUNT(*) AS games,
    ROUND(AVG(f.core_score), 1) AS avg_score,
    ROUND(AVG(f.core_goals), 2) AS avg_goals
FROM fact_player_game f
JOIN dim_event de ON f.event_key = de.event_key
WHERE de.event_tier IS NOT NULL
GROUP BY de.event_tier
ORDER BY avg_score DESC
"""

print('--- Score moyen par event tier ---')
display(pd.read_sql(q_score_tier, engine))

In [ ]:
# Win rate weekend vs semaine
q_weekend = """
SELECT
    CASE WHEN dd.is_weekend THEN 'Weekend' ELSE 'Semaine' END AS period,
    COUNT(*) AS games,
    ROUND(100.0 * SUM(CASE WHEN f.winner THEN 1 ELSE 0 END) / COUNT(*), 2) AS win_rate
FROM fact_player_game f
JOIN dim_date dd ON f.date_key = dd.date_key
GROUP BY dd.is_weekend
ORDER BY period
"""

print('--- Win rate weekend vs semaine ---')
display(pd.read_sql(q_weekend, engine))

In [ ]:
# Top 10 joueurs par rating moyen (min 50 games)
q_top_players = """
SELECT
    dp.player_tag,
    dp.country_name,
    COUNT(*) AS games,
    ROUND(AVG(f.advanced_rating), 3) AS avg_rating,
    ROUND(AVG(f.core_goals), 2) AS avg_goals,
    ROUND(100.0 * SUM(CASE WHEN f.winner THEN 1 ELSE 0 END) / COUNT(*), 1) AS win_rate
FROM fact_player_game f
JOIN dim_player dp ON f.player_key = dp.player_key
GROUP BY dp.player_tag, dp.country_name
HAVING COUNT(*) >= 50
ORDER BY avg_rating DESC
LIMIT 10
"""

print('--- Top 10 joueurs par rating (min 50 games) ---')
display(pd.read_sql(q_top_players, engine))

## 7. Resume

In [ ]:
print('Star schema termine.')
print(f'Base : data/rl.duckdb')
print(f'{df_star["rows"].sum():,} lignes dans 7 tables star')
print(f'  6 dimensions : {sum(dim_counts.values()):,} lignes')
print(f'  1 fact table  : {fact_count:,} lignes ({len([c for c in df_spot.columns])} colonnes verifiees)')
print(f'  FK integrity  : OK (0 orphelins)')